# Diffusion Model Uncertainty Quantification

Diffusion models support uncertainty quantification through repeated stochastic sampling.
Each call to the reverse diffusion process draws a new sample from the learned posterior.
By aggregating N samples for the same sparse sensor input, we estimate:
- **Variance maps**: pixel-wise predictive standard deviation
- **Uncertainty vs. error correlation**: does the model flag where it is wrong?
- **Calibration**: whether higher-uncertainty pixels have higher empirical error

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from torch.utils.data import DataLoader

from data.dataset import GDMDataset
from models.diffusion.model import SimpleUnet
from models.diffusion.noise_scheduler import NoiseScheduler
from models.diffusion.sample import sample_timestep

os.makedirs('figures', exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

diffusion_model = SimpleUnet(dims=(32, 64, 64), in_dim=3).to(DEVICE)
state_dict = torch.load('models/diffusion/weights.pt', map_location=DEVICE, weights_only=False)
diffusion_model.load_state_dict(state_dict)
scheduler = NoiseScheduler(T=100, beta_start=1e-4, beta_end=0.2, device=DEVICE)
diffusion_model.eval()
print('Model loaded.')

In [ ]:
torch.manual_seed(42)
dataset = GDMDataset('../gas-distribution-datasets/synthetic/test/test90.pt', mode='test')
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# Draw one test sample for the qualitative analysis
X, y = next(iter(dataloader))
X, y = X.to(DEVICE), y.to(DEVICE)
print(f'Input shape: {X.shape}  (channels: mask, values)')
print(f'Target shape: {y.shape}')

## 1. Multi-sample generation

The reverse diffusion process injects fresh Gaussian noise at every step `t > 0`
(see `sample_timestep` in `models/diffusion/sample.py`), so each forward pass
through `sample_diffusion_model` for the **same input** yields a different output.
We draw N=32 samples and compute their pixel-wise mean and standard deviation.

In [ ]:
def sample_diffusion_model(X):
    """Draw one sample from the diffusion model posterior for input X."""
    batch_size, _, H, W = X.shape
    x_t = torch.randn((batch_size, 1, H, W), device=DEVICE)
    mask = X[:, 0:1, :, :]
    vals = X[:, 1:2, :, :]
    for t in reversed(range(scheduler.T)):
        t_tensor = torch.full((batch_size,), t, device=DEVICE, dtype=torch.long)
        with torch.no_grad():
            x_t = sample_timestep(x_t, mask, vals, t_tensor, diffusion_model, scheduler)
    return x_t  # [batch_size, 1, H, W]


N_SAMPLES = 32
print(f'Generating {N_SAMPLES} samples for the same input...')
samples = torch.cat([sample_diffusion_model(X) for _ in range(N_SAMPLES)], dim=0)  # [N, 1, 64, 64]
mean_pred = samples.mean(dim=0)   # [1, 64, 64]
std_pred  = samples.std(dim=0)    # [1, 64, 64]
print(f'Done. Samples shape: {samples.shape}')
print(f'Mean predictive std (uncertainty): {std_pred.mean().item():.4f}')

## 2. Variance map

The top row shows the input, ground truth, ensemble mean, and uncertainty (std).
The bottom row illustrates sample diversity: four individual draws from the posterior.
High std indicates regions where the model cannot commit to a single reconstruction.

In [ ]:
def to_np(t):
    return t.squeeze().cpu().detach().numpy()

# Build a shared colormap range from the ground truth
gt_np = to_np(y[0])
vmin, vmax = gt_np.min(), gt_np.max()

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
fig.suptitle('Diffusion Model — Multi-Sample Uncertainty', fontsize=13, y=1.01)

# --- Top row: summary ---
titles_top = ['Input (sensors)', 'Ground Truth', 'Mean Prediction', 'Uncertainty (std)']
images_top = [
    to_np(X[0, 1:2, :, :]),   # sensor values channel
    gt_np,
    to_np(mean_pred),
    to_np(std_pred),
]
cmaps_top = ['viridis', 'viridis', 'viridis', 'hot']
vmins_top = [vmin, vmin, vmin, None]
vmaxs_top = [vmax, vmax, vmax, None]

for col, (title, img, cmap, vn, vx) in enumerate(zip(titles_top, images_top, cmaps_top, vmins_top, vmaxs_top)):
    ax = axes[0, col]
    im = ax.imshow(img, cmap=cmap, vmin=vn, vmax=vx)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Hide unused axes in top row
for col in range(4, 8):
    axes[0, col].axis('off')

# --- Bottom row: individual samples ---
n_show = 8
for col in range(n_show):
    ax = axes[1, col]
    sample_np = to_np(samples[col])
    im = ax.imshow(sample_np, cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_title(f'Sample {col+1}', fontsize=9)
    ax.axis('off')

# Row labels
axes[0, 0].set_ylabel('Summary', fontsize=9, labelpad=4)
axes[1, 0].set_ylabel('Individual draws', fontsize=9, labelpad=4)

plt.tight_layout()
plt.savefig('figures/uncertainty_varmap.pdf', bbox_inches='tight')
plt.show()
print('Saved to figures/uncertainty_varmap.pdf')

## 3. Uncertainty vs. error correlation

For each pixel (excluding observed sensor locations), we compare the predictive standard
deviation to the absolute error between the ensemble mean and ground truth.
A positive Pearson correlation indicates that high-uncertainty regions tend to have
higher reconstruction error — the model is uncertain where it is wrong.

In [ ]:
abs_error = (mean_pred - y[0]).abs()   # [1, 64, 64]

# Exclude sensor locations (the model knows the exact value there)
sensor_mask = X[0, 0:1, :, :]                     # [1, 64, 64], 1 = sensor present
non_sensor  = (sensor_mask == 0).squeeze()         # [64, 64] boolean

std_flat = std_pred.squeeze().cpu().numpy()[non_sensor.cpu().numpy()]
err_flat = abs_error.squeeze().cpu().numpy()[non_sensor.cpu().numpy()]

r = float(np.corrcoef(std_flat, err_flat)[0, 1])

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(std_flat, err_flat, alpha=0.15, s=4, color='steelblue', rasterized=True)

# Trend line
z = np.polyfit(std_flat, err_flat, 1)
x_line = np.linspace(std_flat.min(), std_flat.max(), 100)
ax.plot(x_line, np.poly1d(z)(x_line), color='crimson', linewidth=1.5, label=f'Pearson r = {r:.3f}')

ax.set_xlabel('Predictive Std (uncertainty)', fontsize=11)
ax.set_ylabel('Absolute Error  |mean − truth|', fontsize=11)
ax.set_title('Uncertainty vs. Reconstruction Error\n(non-sensor pixels)', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('figures/uncertainty_scatter.pdf', bbox_inches='tight')
plt.show()
print(f'Pearson r = {r:.3f}  (saved to figures/uncertainty_scatter.pdf)')

## 4. Calibration curve

We aggregate pixels from multiple test samples, bin them by their predictive uncertainty
(std) percentile, and plot the mean absolute error in each bin.
A monotonically increasing curve demonstrates that the model's uncertainty is a reliable
signal — pixels flagged as more uncertain are indeed harder to reconstruct.

In [ ]:
N_CAL  = 20   # number of test inputs
M_DRAW = 16   # diffusion draws per input (fewer than N_SAMPLES for speed)

cal_loader = DataLoader(dataset, batch_size=1, shuffle=True)

all_stds   = []
all_errors = []

for i, (X_cal, y_cal) in enumerate(cal_loader):
    if i >= N_CAL:
        break
    X_cal, y_cal = X_cal.to(DEVICE), y_cal.to(DEVICE)

    cal_samples = torch.cat([sample_diffusion_model(X_cal) for _ in range(M_DRAW)], dim=0)
    cal_mean = cal_samples.mean(dim=0)   # [1, 64, 64]
    cal_std  = cal_samples.std(dim=0)    # [1, 64, 64]
    cal_err  = (cal_mean - y_cal[0]).abs()

    # Exclude sensor locations
    non_sensor_cal = (X_cal[0, 0:1, :, :] == 0).squeeze().cpu().numpy()
    all_stds.append(cal_std.squeeze().cpu().numpy()[non_sensor_cal])
    all_errors.append(cal_err.squeeze().cpu().numpy()[non_sensor_cal])

    if (i + 1) % 5 == 0:
        print(f'  Processed {i+1}/{N_CAL} samples...')

all_stds   = np.concatenate(all_stds)
all_errors = np.concatenate(all_errors)
print(f'Total pixels: {len(all_stds):,}')

In [ ]:
N_BINS = 10
edges  = np.percentile(all_stds, np.linspace(0, 100, N_BINS + 1))

bin_std_med  = []
bin_err_mean = []
bin_err_std  = []

for i in range(N_BINS):
    lo, hi = edges[i], edges[i + 1]
    mask_bin = (all_stds >= lo) & (all_stds <= hi)
    if mask_bin.sum() == 0:
        continue
    bin_std_med.append(np.median(all_stds[mask_bin]))
    bin_err_mean.append(np.mean(all_errors[mask_bin]))
    bin_err_std.append(np.std(all_errors[mask_bin]))

bin_std_med  = np.array(bin_std_med)
bin_err_mean = np.array(bin_err_mean)
bin_err_std  = np.array(bin_err_std)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(bin_std_med, bin_err_mean, 'o-', color='steelblue', linewidth=2, markersize=6)
ax.fill_between(
    bin_std_med,
    bin_err_mean - 0.5 * bin_err_std,
    bin_err_mean + 0.5 * bin_err_std,
    alpha=0.2, color='steelblue', label='±0.5 std'
)
ax.set_xlabel('Predictive Std (bin median)', fontsize=11)
ax.set_ylabel('Mean Absolute Error', fontsize=11)
ax.set_title('Calibration Curve\n(uncertainty bins vs. empirical error)', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/uncertainty_calibration.pdf', bbox_inches='tight')
plt.show()
print('Saved to figures/uncertainty_calibration.pdf')